
# 34 — Newhaven Target Window Selector

Replacement notebook.

This version fixes the plotting bug where the x and y arrays could have different lengths.
It selects the best 4–7 day windows using the best radius and fusion evidence.


In [ ]:

SAT_DAILY_CSV = "outputs/newhaven_multi_radius/satellite_daily_coverage.csv"
FUSION_CSV = "outputs/newhaven_fusion/fusion_daily.csv"
RADIUS_SUMMARY_CSV = "outputs/newhaven_integrated/integrated_radius_summary.csv"
OUTPUT_DIR = "outputs/newhaven_window_selector"
WINDOW_LENGTHS = [4, 5, 6, 7]


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, time, math, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

print("UTC now:", datetime.now(timezone.utc).isoformat())


In [ ]:

sat_daily, sat_meta = safe_read_csv(SAT_DAILY_CSV)
fusion, fusion_meta = safe_read_csv(FUSION_CSV)
radius_summary, radius_meta = safe_read_csv(RADIUS_SUMMARY_CSV)

best_radius = float(radius_summary.iloc[0]["radius_miles"]) if (not radius_summary.empty and "recommended_score" in radius_summary.columns) else 10.0

if not sat_daily.empty and "date" in sat_daily.columns:
    sat_daily["date"] = pd.to_datetime(sat_daily["date"], errors="coerce")
if not fusion.empty and "date" in fusion.columns:
    fusion["date"] = pd.to_datetime(fusion["date"], errors="coerce")

sat_use = sat_daily[sat_daily["radius_miles"].astype(float) == float(best_radius)].copy() if (not sat_daily.empty and "radius_miles" in sat_daily.columns) else pd.DataFrame(columns=["date", "scenes"])

daily = sat_use.merge(fusion, on="date", how="left") if (not sat_use.empty and not fusion.empty) else (sat_use.copy() if not sat_use.empty else fusion.copy())
if daily.empty:
    daily = pd.DataFrame(columns=["date", "scenes", "fusion_score"])

for col in ["scenes", "fusion_score", "article_count", "ground_rows"]:
    if col not in daily.columns:
        daily[col] = 0
    daily[col] = pd.to_numeric(daily[col], errors="coerce").fillna(0)

daily = daily.sort_values("date").reset_index(drop=True)
display(daily.head(20))


In [ ]:

window_rows = []
if not daily.empty and "date" in daily.columns:
    for wlen in WINDOW_LENGTHS:
        for i in range(0, len(daily) - wlen + 1):
            sub = daily.iloc[i:i+wlen].copy()
            if (sub["date"].max() - sub["date"].min()).days != (wlen - 1):
                continue
            window_rows.append({
                "window_days": wlen,
                "start_date": sub["date"].min().date().isoformat(),
                "end_date": sub["date"].max().date().isoformat(),
                "total_scenes": float(sub["scenes"].sum()),
                "mean_fusion_score": float(sub["fusion_score"].mean()),
                "total_article_count": float(sub["article_count"].sum()),
                "total_ground_rows": float(sub["ground_rows"].sum()),
                "window_score": float(sub["scenes"].sum()) * 2.0 + float(sub["fusion_score"].mean()),
                "radius_miles_used": best_radius,
            })

windows = pd.DataFrame(window_rows)
if not windows.empty:
    windows = windows.sort_values(["window_score", "total_scenes", "window_days"], ascending=[False, False, True]).reset_index(drop=True)

windows_csv = Path(OUTPUT_DIR) / "window_candidates.csv"
top_csv = Path(OUTPUT_DIR) / "window_top_candidates.csv"
windows.to_csv(windows_csv, index=False)
windows.head(20).to_csv(top_csv, index=False)

display(windows.head(20))


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))

top_n = min(20, len(windows))
if top_n > 0:
    y = windows.head(top_n)["window_score"].to_numpy()
    x = range(1, top_n + 1)
    ax.plot(x, y, marker="o")

ax.set_title("Top target-window candidate scores")
ax.set_xlabel("Rank")
ax.set_ylabel("Window score")
fig.tight_layout()

plot_path = Path(OUTPUT_DIR) / "window_selector_plot.png"
fig.savefig(plot_path, dpi=150)
plt.show()

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "best_radius_miles": best_radius,
    "inputs": [sat_meta, fusion_meta, radius_meta],
    "rows_windows": int(len(windows)),
    "outputs": {
        "windows_csv": str(windows_csv),
        "top_csv": str(top_csv),
        "plot_png": str(plot_path),
    },
    "notes": [
        "Plotting uses the actual number of rows available, so it remains stable when fewer than 20 window candidates exist."
    ],
}
manifest_path = Path(OUTPUT_DIR) / "window_selector_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Saved:", windows_csv)
print("Saved:", top_csv)
print("Saved:", plot_path)
print("Saved:", manifest_path)
